# Relaxation trajectory visualization

Load saved MLIP relaxation trajectories and visualise how atomic positions, cell parameters, energy, and forces evolve during relaxation.

**Prerequisites:**
1. Run a relaxation with trajectory saving enabled:

```bash
python scripts/relax_batch.py \
    --input  data/candidates/Co-Bi \
    --output data/relaxed/Co-Bi \
    --model  chgnet \
    --fmax   0.05 \
    --max-steps 300 \
    --save-trajectory \
    --save-step-log
```

2. Install visualization dependencies:

```bash
pip install nglview
jupyter nbextension enable nglview --py --sys-prefix
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / "data").is_dir() and (REPO.parent / "data").is_dir():
    REPO = REPO.parent
if not (REPO / "data").is_dir():
    raise FileNotFoundError(
        f"Expected data/ under {REPO}. Open the notebook from the hullgap repo root."
    )

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

print("REPO =", REPO)

## 1. Configuration

In [ ]:
SYSTEM = "Co-Bi"
MODEL = "chgnet"

TRAJ_DIR = REPO / f"data/trajectories/{SYSTEM}"
STEP_CSV = REPO / f"data/results/relaxation_steps_{SYSTEM}_{MODEL}.csv"

print(f"Trajectory dir: {TRAJ_DIR}")
print(f"Step log CSV:   {STEP_CSV}")

## 2. List available trajectories

In [ ]:
from ase.io.trajectory import Trajectory

traj_files = sorted(TRAJ_DIR.glob(f"*_{MODEL}.traj")) if TRAJ_DIR.is_dir() else []

if not traj_files:
    print(
        f"No .traj files found in {TRAJ_DIR}.\n"
        "Run relax_batch.py with --save-trajectory first."
    )
else:
    print(f"{len(traj_files)} trajectory file(s):")
    for tf in traj_files:
        traj = Trajectory(str(tf), mode="r")
        n_frames = len(traj)
        formula = traj[0].get_chemical_formula(mode="metal")
        traj.close()
        print(f"  {tf.name:45s}  {formula:8s}  {n_frames} frames")

## 3. Pick a candidate to visualize

Set `CANDIDATE` to the stem of one of the trajectory files above (without the `_<model>.traj` suffix).

In [ ]:
# Auto-select the first trajectory, or set manually
if traj_files:
    CANDIDATE = traj_files[0].stem.removesuffix(f"_{MODEL}")
else:
    CANDIDATE = "candidate_demo_001"  # fallback

traj_path = TRAJ_DIR / f"{CANDIDATE}_{MODEL}.traj"
print(f"Selected: {CANDIDATE}")
print(f"Trajectory: {traj_path}")

## 4. Load trajectory

In [ ]:
traj = Trajectory(str(traj_path), mode="r")
frames = list(traj)
traj.close()

print(f"{len(frames)} frames")
print(f"Initial formula: {frames[0].get_chemical_formula(mode='metal')}")
print(f"Initial cell:    {frames[0].cell.cellpar()[:3].round(3)} \u00c5")
print(f"Final cell:      {frames[-1].cell.cellpar()[:3].round(3)} \u00c5")

## 5. 3D trajectory replay (nglview)

Use the slider to step through the relaxation. If nglview is not installed, this cell is skipped.

In [ ]:
try:
    import nglview as nv

    view = nv.show_asetraj(frames)
    view.add_unitcell()
    view.add_spacefill(radius_type="covalent", scale=0.5)
    view.remove_ball_and_stick()
    view.camera = "orthographic"
    view
except ImportError:
    print(
        "nglview is not installed. Install it for interactive 3D visualization:\n"
        "  pip install nglview\n"
        "  jupyter nbextension enable nglview --py --sys-prefix"
    )

## 6. Per-step convergence plots

Plot energy, forces, volume, and cell parameters vs. relaxation step.

In [ ]:
if STEP_CSV.is_file():
    step_df = pd.read_csv(STEP_CSV)
    cand_steps = step_df[step_df["candidate_id"] == CANDIDATE].copy()
else:
    cand_steps = pd.DataFrame()

if cand_steps.empty:
    # Reconstruct from trajectory frames
    rows = []
    for i, atoms in enumerate(frames):
        e = atoms.get_potential_energy()
        f = atoms.get_forces()
        fmax = float(np.max(np.linalg.norm(f, axis=1)))
        vol = atoms.get_volume()
        cp = atoms.cell.cellpar()
        rows.append({
            "step": i,
            "energy_eV": e,
            "energy_per_atom_eV": e / len(atoms),
            "fmax_eV_A": fmax,
            "volume_A3": vol,
            "volume_per_atom_A3": vol / len(atoms),
            "a_A": cp[0], "b_A": cp[1], "c_A": cp[2],
            "alpha_deg": cp[3], "beta_deg": cp[4], "gamma_deg": cp[5],
        })
    cand_steps = pd.DataFrame(rows)
    print("Step log reconstructed from trajectory frames.")
else:
    print(f"Loaded {len(cand_steps)} steps from {STEP_CSV.name}")

cand_steps.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
s = cand_steps

# Energy
ax = axes[0, 0]
ax.plot(s["step"], s["energy_per_atom_eV"], "o-", ms=4, color="#2c3e50")
ax.set_xlabel("Step")
ax.set_ylabel("Energy / atom (eV)")
ax.set_title("Energy convergence")

# Forces
ax = axes[0, 1]
ax.plot(s["step"], s["fmax_eV_A"], "o-", ms=4, color="#e74c3c")
ax.axhline(0.05, color="gray", ls="--", lw=1, label="fmax=0.05")
ax.set_xlabel("Step")
ax.set_ylabel("Max force (eV/\u00c5)")
ax.set_title("Force convergence")
ax.set_yscale("log")
ax.legend()

# Volume
ax = axes[1, 0]
ax.plot(s["step"], s["volume_per_atom_A3"], "o-", ms=4, color="#27ae60")
ax.set_xlabel("Step")
ax.set_ylabel("Volume / atom (\u00c5\u00b3)")
ax.set_title("Volume evolution")

# Cell parameters
ax = axes[1, 1]
for lbl, col, c in [("a", "a_A", "#3498db"), ("b", "b_A", "#e67e22"), ("c", "c_A", "#9b59b6")]:
    ax.plot(s["step"], s[col], "o-", ms=4, color=c, label=lbl)
ax.set_xlabel("Step")
ax.set_ylabel("Lattice parameter (\u00c5)")
ax.set_title("Cell parameter evolution")
ax.legend()

fig.suptitle(f"{CANDIDATE} ({MODEL}) — relaxation trajectory", fontsize=14, y=1.02)
plt.tight_layout()

fig_path = REPO / f"reports/figures/{CANDIDATE}_{MODEL}_convergence.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {fig_path}")

## 7. Compare convergence across candidates

Overlay energy convergence curves for all candidates in the step log.

In [ ]:
if STEP_CSV.is_file():
    full_df = pd.read_csv(STEP_CSV)
else:
    full_df = cand_steps.copy()
    full_df["candidate_id"] = CANDIDATE

candidates = full_df["candidate_id"].unique()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for cid in candidates:
    sub = full_df[full_df["candidate_id"] == cid]
    ax1.plot(sub["step"], sub["energy_per_atom_eV"], "o-", ms=3, label=cid)
    ax2.plot(sub["step"], sub["fmax_eV_A"], "o-", ms=3, label=cid)

ax1.set_xlabel("Step")
ax1.set_ylabel("Energy / atom (eV)")
ax1.set_title("Energy convergence")
ax1.legend(fontsize=7, loc="best")

ax2.set_xlabel("Step")
ax2.set_ylabel("Max force (eV/\u00c5)")
ax2.set_title("Force convergence")
ax2.set_yscale("log")
ax2.axhline(0.05, color="gray", ls="--", lw=1)
ax2.legend(fontsize=7, loc="best")

fig.suptitle(f"{SYSTEM} ({MODEL}) — all candidates", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Initial vs. final structure side-by-side

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pymatgen.io.ase import AseAtomsAdaptor

element_colors = {
    "Co": "#3366cc", "Bi": "#cc6633", "Fe": "#e74c3c",
    "Ni": "#2ecc71", "Mn": "#9b59b6", "Sb": "#e67e22",
}

def plot_structure_3d(ax, atoms, title):
    struct = AseAtomsAdaptor.get_structure(atoms)
    species = [str(s.specie) for s in struct]
    fc = struct.frac_coords
    for sym in set(species):
        mask = [s == sym for s in species]
        pts = fc[mask]
        ax.scatter(
            pts[:, 0], pts[:, 1], pts[:, 2],
            s=120, label=sym,
            c=element_colors.get(sym, "#444444"),
        )
    ax.set_xlabel("x (frac)")
    ax.set_ylabel("y (frac)")
    ax.set_zlabel("z (frac)")
    ax.set_title(title)
    ax.legend(fontsize=8)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")

plot_structure_3d(ax1, frames[0], f"Initial (step 0)")
plot_structure_3d(ax2, frames[-1], f"Final (step {len(frames)-1})")

fig.suptitle(f"{CANDIDATE} ({MODEL})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()